In [ ]:
%load_ext autoreload
%autoreload 2


import pandas as pd
import numpy as np
import os
import pydicom
import nibabel as nib
import dicom2nifti
import SimpleITK as sitk

from glob import glob
from pathlib import Path
import json
import os


def make_df_from_folder(base_dcm_folder = "/home/clemens/data/RADIPOP_EXTRA/FINAL SEGM DATEN"):
    dcm_folders = glob(os.path.join(base_dcm_folder, "*"))
    # For each folder, find the first DICOM file, read its header, and extract the patient name
    patient_names = []
    for folder in dcm_folders:
        # Find the first .dcm file in the folder (recursively, just in case)
        dcm_files = []
        for root, _, files in os.walk(folder):
            dcm_files += [os.path.join(root, f) for f in files if f.lower().endswith(".dcm")]
            if dcm_files:
                break  # Only need the first found .dcm in the entire folder tree
        if dcm_files:
            dcm_path = dcm_files[0]
            try:
                ds = pydicom.dcmread(dcm_path, stop_before_pixels=True)
                # Try to get PatientName in a printable string form
                patient_name = str(ds.get("PatientName", "Unknown"))
            except Exception as e:
                patient_name = f"Error: {e}"
        else:
            patient_name = "No DICOM file found"
        patient_names.append({"folder": folder, "patient_name": patient_name})

    # Display the results as a DataFrame
    df_names = pd.DataFrame(patient_names)

    df_names["ID"] = df_names["folder"].str.split("/").str[-1]




    # For collecting the detailed info
    detailed_rows = []

    for folder in dcm_folders:
        patient_id = os.path.basename(folder)
        by_acqtime_folder = os.path.join(folder, "by_acqtime")
        if not os.path.isdir(by_acqtime_folder):
            # No by_acqtime folder, skip
            continue
        acq_time_folders = [f for f in os.listdir(by_acqtime_folder) 
                            if os.path.isdir(os.path.join(by_acqtime_folder, f))]
        num_acqs = len(acq_time_folders)
        for acq_time in acq_time_folders:
            acq_folder_path = os.path.join(by_acqtime_folder, acq_time)
            # Count number of subfolders in this acq_time folder
            subfolders = [sf for sf in os.listdir(acq_folder_path)
                        if os.path.isdir(os.path.join(acq_folder_path, sf))]
            num_subfolders = len(subfolders)
            # Get contrast_phase.json content
            contrast_path = os.path.join(acq_folder_path, "contrast_phase.json")
            contrast_json = None
            phase = None
            pi_time = None
            phase_probabilities = None
            error_msg = None
            if os.path.isfile(contrast_path):
                try:
                    with open(contrast_path, "r") as f:
                        contrast_json = json.load(f)
                    # Unpack the keys if they exist; use None if key is missing
                    phase = contrast_json.get("phase") if isinstance(contrast_json, dict) else None
                    pi_time = contrast_json.get("pi_time") if isinstance(contrast_json, dict) else None
                    # Unpack phase probabilities if present
                    if isinstance(contrast_json, dict):
                        phase_probabilities = contrast_json.get("probabilities")
                        # If phase_probabilities is a dictionary, flatten to individual columns
                        if isinstance(phase_probabilities, dict):
                            for prob_key, prob_val in phase_probabilities.items():
                                # Add each probability as its own variable below (they get added to the row)
                                pass
                except Exception as e:
                    error_msg = f"Error: {e}"
            detailed_row = {
                "folder": folder,
                "ID": patient_id,
                "acq_time": acq_time,
                #"num_subfolders": num_subfolders,
                "num_unique_acq_times": num_acqs,
            }
            # Add the unpacked (or missing/None) values to the row
            detailed_row["contrast_phase_phase"] = phase
            detailed_row["contrast_phase_pi_time"] = pi_time
            if isinstance(phase_probabilities, dict):
                for prob_key, prob_val in phase_probabilities.items():
                    detailed_row[f"contrast_phase_prob_{prob_key}"] = prob_val
            if error_msg is not None:
                detailed_row["contrast_phase_error"] = error_msg
            detailed_rows.append(detailed_row)

    df_acq = pd.DataFrame(detailed_rows)


    df_merged = df_acq.merge(df_names, left_on="ID", right_on="ID", how="left")

    m = (df_merged["contrast_phase_phase"] == "portal_venous")

    df_pv = df_merged[m]

    # Count how many times each ID appears in df_pv and add as a new column
    df_pv["ID_count_in_df_pv"] = df_pv.groupby("ID")["ID"].transform("count")

    df_pv.sort_values(["ID_count_in_df_pv", "ID"], ascending=[False, True])
    assert (df_pv["folder_x"] == df_pv["folder_y"]).all(), "folder_x and folder_y are not identical!"
    df_pv = df_pv.drop(columns=["folder_y"]).rename(columns={"folder_x": "folder"})

    # Use os.path.join for correct path construction (avoid string concatenation)
    path_to_features = df_pv.apply(lambda row: os.path.join(row["folder"], "by_acqtime", str(row["acq_time"]), str(row["acq_time"])), axis=1)
    missing_files = []
    for folder in path_to_features:
        liver_path = os.path.join(folder, "Features_liver.xlsx")
        spleen_path = os.path.join(folder, "Features_spleen.xlsx")
        # Debug: Print each path before checking
        if not os.path.isfile(liver_path):
            missing_files.append(liver_path)
        if not os.path.isfile(spleen_path):
            missing_files.append(spleen_path)

    if missing_files:
        print("Some files are missing! First 5 missing files:")
        print(missing_files[:5])

    assert len(missing_files) == 0, f"Missing feature files {len(missing_files)}: {missing_files}"
    df_pv["path_to_features"] = path_to_features
    return df_pv


In [ ]:

df_pv = make_df_from_folder()


In [ ]:

def add_predictions_to_df(df_pv, prefix):
    results = []
    for p in df_pv["path_to_features"]:
        json_file = os.path.join(p, f"predictions_{prefix}.json")
        try:
            with open(json_file, "r") as f:
                predictions = json.load(f)
            unpacked = {f"predictions_{prefix}: {k}": v for k, v in predictions.items()}
        except Exception:
            unpacked = {}
        results.append(unpacked)
    results_df = pd.DataFrame(results)
    df_pv = pd.concat([df_pv.reset_index(drop=True), results_df.reset_index(drop=True)], axis=1)
    return df_pv

df_pv = add_predictions_to_df(df_pv, "ORD")
df_pv = add_predictions_to_df(df_pv, "WFD")


df_pv.columns


In [ ]:
df_pv["surname"] = df_pv["patient_name"].apply(lambda x: x.split("^")[0].upper())
df_pv["first_name"] = df_pv["patient_name"].apply(lambda x: x.split("^")[1].upper())
df_pv["fn_ln"] = df_pv["first_name"] + " " + df_pv["surname"]


In [ ]:
# Load HVPG data
src="/home/clemens/data/RADIPOP_EXTRA/tabular_data/Beaujon cohort complete list included patients_mod_3_add_HCC.xlsx"
df = pd.read_excel(src)[["Name", "First name", "HVPG", "DOB"]]
df["surname"] = df["Name"].apply(lambda x: x.upper())
df["first_name"] = df["First name"].apply(lambda x: x.upper())
df["fn_ln"] = df["first_name"] + " " + df["surname"]





In [ ]:
# Check for non-unique matches in df before merging
df_duplicates = df[df.duplicated(subset=["fn_ln"], keep=False)]
if len(df_duplicates) > 0:
    print(f"WARNING: Found {len(df_duplicates)} non-unique matches in df on fn_ln:")
    print(df_duplicates[["fn_ln", "Name", "First name", "HVPG", "DOB"]].sort_values("fn_ln"))
    print("\n")

# Perform left merge
df_merged = df_pv.merge(df, on="fn_ln", how="left", indicator=True)

# Check for unmatched rows
unmatched = df_merged[df_merged["_merge"] == "left_only"]
if len(unmatched) > 0:
    print(f"WARNING: Found {len(unmatched)} rows in df_pv with no match in df:")
    print(unmatched[["fn_ln", "patient_name", "ID"]].head(20))
    if len(unmatched) > 20:
        print(f"... and {len(unmatched) - 20} more unmatched rows")
    print("\n")

# Check for multiple matches (shouldn't happen if df is unique, but check anyway)
if len(df_duplicates) > 0:
    # Find rows in df_merged that matched to duplicate fn_ln values
    matched_to_duplicates = df_merged[df_merged["fn_ln"].isin(df_duplicates["fn_ln"]) & (df_merged["_merge"] == "both")]
    if len(matched_to_duplicates) > 0:
        print(f"WARNING: Found {len(matched_to_duplicates)} rows in df_merged that matched to non-unique fn_ln values:")
        print(matched_to_duplicates[["fn_ln", "patient_name", "ID", "HVPG"]].head(20))
        if len(matched_to_duplicates) > 20:
            print(f"... and {len(matched_to_duplicates) - 20} more rows")
        print("\n")

# Drop the merge indicator column
df_merged = df_merged.drop(columns=["_merge"])

print(f"Merge complete. df_merged has {len(df_merged)} rows (df_pv had {len(df_pv)} rows)")
df_merged.head()


In [ ]:
#df_merged.sort_values("Name", ascending=False).to_excel("/home/clemens/data/RADIPOP_EXTRA/predictions_totalSegmentatior_auto_merged.xlsx", index=False)


m = ~df_merged["HVPG"].isna()

df_ = df_merged[m]


(df_["predictions_WFD: RF_HVPG"] - df_["HVPG"]).abs().describe()




In [ ]:
## Print the script to run the inference
if False:    
    script_string = ""

    model_dir = "/home/clemens/data/RADIOMICS_LOCAL/model_dir/radipop_no_autoseg_spacing_111_WFD_original_params"
    out_name="predictions_WFD"
    script_string += f'model_dir="{model_dir}"\n\n\n'
    script_string += f'out_name="{out_name}"\n\n\n'
    for p in path_to_features:
        # Use shell bash variable reference, don't f-string the braces!
        s = f'python -m radipop_utils.tools.inference_from_features_entry_point --features_folder "{p}" --model_dir "$model_dir" --out_name "$out_name"'
        script_string += f"{s}\n"

    print(script_string)
